# MQ-009: Medical RAG Retrieval Evaluation

## Research Objective

Evaluate semantic retrieval quality using the
MedQuAD medical knowledge corpus and FAISS.

## Previously Completed

- Cleaned medical records: 15,864
- Final medical passages: 38,544
- Embedding model: all-MiniLM-L6-v2
- Embedding dimensions: 384
- Vector index: FAISS IndexFlatIP
- Similarity: Cosine-equivalent inner product

## Notebook Scope

This notebook loads previously generated artifacts.

It does not repeat:
- Medical data preprocessing
- Passage chunking
- Embedding generation
- FAISS index construction

## Evaluation Plan

1. Load saved retrieval artifacts.
2. Load held-out validation questions.
3. Perform semantic retrieval.
4. Evaluate retrieval relevance.
5. Record experimental findings.

In [1]:
from pathlib import Path

import faiss
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    "/home/sagemaker-user/medical_chatbot"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load passage metadata
passages_df = pd.read_csv(
    PROCESSED_DIR / "medquad_final_passages.csv"
)

# Load existing FAISS index
faiss_index = faiss.read_index(
    str(PROCESSED_DIR / "medquad_faiss.index")
)

# Validate artifact consistency
assert len(passages_df) == faiss_index.ntotal
assert faiss_index.d == 384

print("Medical passages:", len(passages_df))
print("FAISS indexed vectors:", faiss_index.ntotal)
print("Embedding dimensions:", faiss_index.d)
print("Artifacts loaded successfully!")

Medical passages: 38544
FAISS indexed vectors: 38544
Embedding dimensions: 384
Artifacts loaded successfully!


In [2]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu"
)

print(
    "Embedding dimensions:",
    embedding_model.get_embedding_dimension()
)

print("Query embedding model ready!")

/home/sagemaker-user/medquad-envs/rag-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3012.33it/s]


Embedding dimensions: 384
Query embedding model ready!


In [3]:
def retrieve_medical_passages(query, top_k=5):

    # Encode only the new query
    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cpu"
    ).astype(np.float32)

    # Search existing FAISS index
    scores, indices = faiss_index.search(
        query_vector,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if idx < 0:
            continue

        record = passages_df.iloc[int(idx)]

        results.append({
            "rank": rank,
            "similarity": float(score),
            "question": record["question"],
            "passage": record["passage"],
            "source": record["source"],
            "focus_area": record["focus_area"]
        })

    return pd.DataFrame(results)

In [4]:
query = "What are the symptoms of chronic kidney disease?"

results = retrieve_medical_passages(
    query,
    top_k=5
)

display(
    results[
        [
            "rank",
            "similarity",
            "question",
            "source",
            "focus_area"
        ]
    ]
)

,rank,similarity,question,source,focus_area
0,1,0.715603,What are the symptoms of Kidney Disease ?,NIHSeniorHealth,Kidney Disease
1,2,0.686266,What is (are) Chronic Kidney Disease ?,MPlusHealthTopics,Chronic Kidney Disease
2,3,0.677963,What are the symptoms of Atherosclerosis ?,NHLBI,Atherosclerosis
3,4,0.676642,What are the symptoms of Kidney Disease ?,NIHSeniorHealth,Kidney Disease
4,5,0.665614,What are the complications of Polycystic Kidne...,NIDDK,Polycystic Kidney Disease


## MQ-009: Retrieved Evidence Inspection

### Objective
Assess whether retrieved medical passages directly
answer the user's question.

### Evaluation Criteria
- Medical condition relevance
- Question-answer relevance
- Presence of useful clinical information
- Potentially misleading or unrelated evidence

### Research Importance
High embedding similarity does not necessarily imply
high clinical relevance.

The inspection will help identify whether additional
retrieval filtering or reranking is needed.

In [5]:
query = "What are the symptoms of chronic kidney disease?"

results = retrieve_medical_passages(query, top_k=5)

for row in results.itertuples(index=False):

    print(f"\nRank: {row.rank}")
    print(f"Similarity: {row.similarity:.4f}")
    print(f"Source: {row.source}")
    print(f"Focus area: {row.focus_area}")
    print(f"Original question: {row.question}")

    print("\nMedical evidence:")
    print(row.passage)

    print("\n" + "=" * 80)


Rank: 1
Similarity: 0.7156
Source: NIHSeniorHealth
Focus area: Kidney Disease
Original question: What are the symptoms of Kidney Disease ?

Medical evidence:
appetite, and some have leg cramps. These problems are caused by waste products that build up in the blood, a condition known as uremia. Healthy kidneys remove waste products from the blood. When the kidneys stop working, uremia occurs. The kidneys also make hormones and balance the minerals in the blood. When the kidneys stop working, most people develop conditions that affect the blood, bones, nerves, and skin. These problems may include itching, sleep problems, restless legs, weak bones, joint problems, and depression. How Kidney Disease Is Diagnosed Blood and urine tests are the only way to check for kidney damage or measure kidney function. It is important for you to get checked for kidney disease if you have the


Rank: 2
Similarity: 0.6863
Source: MPlusHealthTopics
Focus area: Chronic Kidney Disease
Original question: What

In [6]:
from pathlib import Path

# Retrieve baseline results
query = "What are the symptoms of chronic kidney disease?"

results = retrieve_medical_passages(
    query,
    top_k=5
)

# Create evaluation output directory
output_dir = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "data/processed/retrieval_evaluation"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Save full passages without truncation
output_path = output_dir / "ckd_baseline_retrieval.csv"

results.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Retrieved passages:", len(results))

# Display one passage at a time to avoid output truncation
for _, row in results.iterrows():
    print(
        f"Rank {row['rank']} | "
        f"Similarity {row['similarity']:.4f} | "
        f"{row['focus_area']}"
    )
    print("Passage length:", len(row["passage"]))

Saved: /home/sagemaker-user/medical_chatbot/data/processed/retrieval_evaluation/ckd_baseline_retrieval.csv
Retrieved passages: 5
Rank 1 | Similarity 0.7156 | Kidney Disease
Passage length: 727
Rank 2 | Similarity 0.6863 | Chronic Kidney Disease
Passage length: 689
Rank 3 | Similarity 0.6780 | Atherosclerosis
Passage length: 754
Rank 4 | Similarity 0.6766 | Kidney Disease
Passage length: 719
Rank 5 | Similarity 0.6656 | Polycystic Kidney Disease
Passage length: 677


## MQ-010: Held-Out Retrieval Evaluation

### Objective
Evaluate the baseline FAISS retriever using 50 held-out
medical questions.

### Dataset
- Validation dataset: validation.csv
- Storage: Amazon S3
- Retrieval corpus: 38,544 indexed passages

### Evaluation Strategy
The validation questions will be encoded using the
existing sentence-transformer model.

Retrieved passages will be assessed for medical topic
relevance and answer usefulness.

### Research Limitations
A matching medical focus area does not guarantee that
a passage correctly answers the question.

Human relevance judgments or independently verified
reference evidence will be needed for stronger
retrieval-quality conclusions.